<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/OpenVoice-V2_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OpenVoice V2 — Instant Multilingual Voice Cloning & Conversion

OpenVoice is the open-source voice-cloning / voice-conversion model from [MIT + MyShell](https://research.myshell.ai/open-voice). It clones the **tone color** (timbre / accent / rhythm / emotion) of a short reference clip and applies it to **any source audio**, with native multi-lingual support (English, Spanish, French, Chinese, Japanese, Korean) via MeloTTS as the base speaker. **MIT licensed — free for commercial use.**

> "OpenVoice can accurately clone the reference tone color and generate speech in multiple languages and accents." — [arXiv 2312.01479](https://arxiv.org/abs/2312.01479)

**Upstream**: [myshell-ai/OpenVoice](https://github.com/myshell-ai/OpenVoice) · [myshell-ai/OpenVoiceV2](https://huggingface.co/myshell-ai/OpenVoiceV2) · [arXiv 2312.01479](https://arxiv.org/abs/2312.01479) · [MIT](https://github.com/myshell-ai/OpenVoice/blob/main/LICENSE)

## What this notebook does

- **Voice conversion** — change the *timbre* of an audio clip to match a short reference voice (5–30 s), keeping the *content / prosody* of the source
- **Text-to-speech with cloned voice** — type text in EN/ES/FR/ZH/JA/KR, get it spoken in the reference voice
- **Tone color control** — adjust emotion, accent, rhythm, and pitch via the `tau` slider
- **Watermarking** — every output is watermarked with an embedded message (default `@MyShell`) via `wavmark`
- **Six tabs**: Convert, TTS+Convert, Style Controls, Batch, VRAM, Help
- **Two model versions in one notebook** — V1 (simpler, EN/ZH only) and V2 (multilingual, better quality)

## What this notebook does NOT do

- **Real-time streaming** voice conversion (Seed-VC does this but is GPL-3.0, see Seed-VC roadmap)
- **Training a new base speaker** — only zero-shot cloning from reference audio
- **Transcription / ASR** — the source audio's words are preserved as-is

## How to use

1. Run **STEP 1** (installs OpenVoice + MeloTTS + Japanese dictionary + ffmpeg).
2. *Optional* — run **STEP 2** to pre-cache checkpoints to Drive (~300 MB for V1+V2).
3. Run **STEP 3** to load the core functions.
4. Run **STEP 4** for the interactive Gradio UI.
5. *Optional* — run **STEP 5** to prevent Colab from disconnecting.
6. *Optional* — run **STEP 6** for a one-off conversion, or **STEP 7** for batch.

> **Tip:** First call downloads a small faster-whisper model for VAD (~1 GB). Set `vad=False` in `extract_target_se` to skip it (and use the whole ref clip as-is).

## Citation

```bibtex
@article{qin2023openvoice,
  title  = {OpenVoice: Versatile Instant Voice Cloning},
  author = {Qin, Zengyi and Zhao, Wenliang and Yu, Xumin and Sun, Xin},
  journal= {arXiv preprint arXiv:2312.01479},
  year   = {2023}
}
```


In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs the `openvoice` package (with bundled MeloTTS for V2 multilingual), `unidic` for Japanese G2P, `ffmpeg` (system), and `wavmark` for watermarking. ~5 min first time.

import os, sys, subprocess, time

# GPU auto-detect
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[GPU] torch {torch.__version__} · device = {DEVICE}', flush=True)
if DEVICE == 'cuda':
    try:
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'[GPU] {name} ({vram:.1f} GB VRAM)', flush=True)
    except Exception as e:
        print(f'[GPU] (could not read device name: {e})', flush=True)

# Drive cache setup — OpenVoice checkpoints are ~300 MB total for V1+V2
DRIVE = '/content/drive/MyDrive/AEI_TTS_Cache/huggingface'
os.environ.setdefault('HF_HOME', DRIVE)
os.makedirs(DRIVE, exist_ok=True)
print(f'[Cache] HF_HOME = {DRIVE}', flush=True)

# Output dirs
os.makedirs('/content/voice_out', exist_ok=True)
os.makedirs('/content/ref_audio', exist_ok=True)
print('[Setup] /content/voice_out and /content/ref_audio ready.', flush=True)

# System package: ffmpeg (required by pydub)
print('\n[apt] Installing ffmpeg...', flush=True)
subprocess.run(['apt-get', '-qq', '-y', 'install', 'ffmpeg'],
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
import shutil
print(f'[apt] ffmpeg = {shutil.which("ffmpeg")}', flush=True)

# Install openvoice (VITS + tone color converter + V1/V2 checkpoints downloader)
# We pin numpy more permissively than the upstream setup.py so it co-exists with torch 2.x.
print('\n[pip] Installing openvoice (git) — ~3-5 min...', flush=True)
t0 = time.time()
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'numpy<2.0', 'cython',
     'git+https://github.com/myshell-ai/OpenVoice.git',
     'git+https://github.com/myshell-ai/MeloTTS.git',
     'wavmark', 'pydub', 'librosa', 'inflect', 'gradio>=4.0', 'tqdm', 'hf_transfer'],
    check=True,
)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print(f'[pip] Done in {time.time()-t0:.1f}s', flush=True)

# Japanese G2P dictionary (unidic) for MeloTTS
print('\n[unidic] Downloading Japanese dictionary...', flush=True)
t0 = time.time()
try:
    import unidic
    subprocess.run([sys.executable, '-m', 'unidic', 'download'],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    print(f'[unidic] Done in {time.time()-t0:.1f}s', flush=True)
except Exception as e:
    print(f'[unidic] skipped: {e} (only needed for JP language)')

# Sanity-check the imports
try:
    import openvoice
    from openvoice.api import ToneColorConverter
    from openvoice import se_extractor
    print(f'[openvoice] imported OK', flush=True)
except Exception as e:
    print(f'[openvoice] FAILED: {e}', flush=True)
    raise

try:
    from melo.api import TTS as MeloTTS
    print('[melo] imported OK', flush=True)
except Exception as e:
    print(f'[melo] FAILED: {e}', flush=True)
    raise

print('\n[Done] STEP 1 complete. Move on to STEP 2 (pre-cache) or STEP 3 (core).', flush=True)


In [ ]:
# @title STEP 2 — Pre-cache Model Checkpoints
# @markdown Downloads the OpenVoice V1 + V2 checkpoints from the MyShell S3 bucket to the local cache (or Drive if mounted). Resumable.

# @markdown **V1 size: ~150 MB · V2 size: ~140 MB · MeloTTS models: ~150 MB each, downloaded on demand per language.**

import os, urllib.request, zipfile, shutil, time

CACHE_ROOT = '/content/openvoice_models'
os.makedirs(CACHE_ROOT, exist_ok=True)

# V1 checkpoint
V1_URL = 'https://myshell-public-repo-host.s3.amazonaws.com/openvoice/checkpoints_1226.zip'
V1_DIR = os.path.join(CACHE_ROOT, 'checkpoints')
V1_ZIP = os.path.join(CACHE_ROOT, 'checkpoints_1226.zip')

# V2 checkpoint
V2_URL = 'https://myshell-public-repo-hosting.s3.amazonaws.com/openvoice/checkpoints_v2_0417.zip'
V2_DIR = os.path.join(CACHE_ROOT, 'checkpoints_v2')
V2_ZIP = os.path.join(CACHE_ROOT, 'checkpoints_v2_0417.zip')


def _download_with_progress(url, dst):
    if os.path.exists(dst) and os.path.getsize(dst) > 1_000_000:
        print(f'  [skip] {dst} already exists ({os.path.getsize(dst)/1e6:.1f} MB)', flush=True)
        return
    print(f'  ↓ Downloading {url}', flush=True)
    t0 = time.time()
    urllib.request.urlretrieve(url, dst)
    print(f'  ✓ {os.path.getsize(dst)/1e6:.1f} MB in {time.time()-t0:.1f}s', flush=True)


def _extract_if_needed(zip_path, target_dir):
    if os.path.isdir(target_dir) and os.listdir(target_dir):
        print(f'  [skip] {target_dir} already extracted', flush=True)
        return
    print(f'  ↓ Extracting {zip_path}...', flush=True)
    t0 = time.time()
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(CACHE_ROOT)
    print(f'  ✓ extracted in {time.time()-t0:.1f}s', flush=True)


print(f'[Cache] {CACHE_ROOT}\n')

# V1
print('[V1] checkpoints_1226.zip ...')
_download_with_progress(V1_URL, V1_ZIP)
_extract_if_needed(V1_ZIP, V1_DIR)
if os.path.isdir(V1_DIR):
    print(f'  V1 contents: {sorted(os.listdir(V1_DIR))}')

# V2
print('\n[V2] checkpoints_v2_0417.zip ...')
_download_with_progress(V2_URL, V2_ZIP)
_extract_if_needed(V2_ZIP, V2_DIR)
if os.path.isdir(V2_DIR):
    print(f'  V2 contents: {sorted(os.listdir(V2_DIR))}')

# Cleanup zips to save disk
for z in (V1_ZIP, V2_ZIP):
    if os.path.exists(z) and os.path.isdir(V1_DIR) and os.path.isdir(V2_DIR):
        try:
            os.remove(z)
            print(f'  [clean] removed {z}', flush=True)
        except Exception:
            pass

print('\n[Done] STEP 2 complete. V1 and V2 checkpoints cached.', flush=True)


In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Defines the inference wrappers. The V1/V2 tone-color converters and MeloTTS base speakers are loaded on first use, not at import time. This cell is fast.

import os, time, gc
import numpy as np
import torch
import soundfile as sf
from typing import Optional, Tuple

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAMPLE_RATE = 22050  # OpenVoice output rate (matches VITS internal)

# ── Language support table ────────────────────────────────────────
# (melo_lang_code, display label, base_se_filename_stem)
MELO_LANGS = [('EN_NEWEST', 'English 🇺🇸 (newest)', 'en_newest'), ('EN', 'English 🇺🇸 (default)', 'en-default'), ('ES', 'Spanish 🇪🇸', 'es'), ('FR', 'French 🇫🇷', 'fr'), ('ZH', 'Chinese 🇨🇳 (mixed EN)', 'zh'), ('JP', 'Japanese 🇯🇵', 'jp'), ('KR', 'Korean 🇰🇷', 'kr')]

CACHE_ROOT = '/content/openvoice_models'
V1_CKPT_DIR = os.path.join(CACHE_ROOT, 'checkpoints')
V2_CKPT_DIR = os.path.join(CACHE_ROOT, 'checkpoints_v2')

# ── Lazy converter cache (V1 and V2 separately) ─────────────────
CONVERTERS = {}  # version -> ToneColorConverter


def get_converter(version: str = 'v2'):
    """Lazily build a ToneColorConverter for the given OpenVoice version.
    version: 'v1' (simpler, EN/ZH only) | 'v2' (multilingual via MeloTTS)
    """
    version = version.lower()
    if version in CONVERTERS and CONVERTERS[version] is not None:
        return CONVERTERS[version]
    from openvoice.api import ToneColorConverter
    if version == 'v1':
        ckpt_dir = V1_CKPT_DIR
    elif version == 'v2':
        ckpt_dir = V2_CKPT_DIR
    else:
        raise ValueError(f"version must be 'v1' or 'v2', got {version!r}")
    print(f'[Converter] {version} from {ckpt_dir}...', flush=True)
    t0 = time.time()
    conv = ToneColorConverter(f'{ckpt_dir}/converter/config.json', device=DEVICE)
    conv.load_ckpt(f'{ckpt_dir}/converter/checkpoint.pth')
    CONVERTERS[version] = conv
    print(f'[Converter] {version} ready in {time.time()-t0:.1f}s', flush=True)
    return conv


def free_converter(version: str = None):
    """Release one or all converters from VRAM."""
    if version is None:
        for v in list(CONVERTERS.keys()):
            free_converter(v)
        return
    if version in CONVERTERS and CONVERTERS[version] is not None:
        del CONVERTERS[version]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'[Free] {version!r} converter released', flush=True)


# ── Lazy MeloTTS cache (one per language) ────────────────────────
MELO = {}  # lang_code -> MeloTTS object


def get_melo(lang_code: str = 'EN'):
    """Lazily load MeloTTS for a language. Auto-downloads on first call (~150 MB)."""
    if lang_code in MELO and MELO[lang_code] is not None:
        return MELO[lang_code]
    from melo.api import TTS as MeloTTS
    print(f'[MeloTTS] {lang_code} (device={DEVICE}) ...', flush=True)
    t0 = time.time()
    model = MeloTTS(language=lang_code, device=DEVICE)
    MELO[lang_code] = model
    print(f'[MeloTTS] {lang_code} ready in {time.time()-t0:.1f}s', flush=True)
    return model


def free_melo(lang_code: str = None):
    """Release one or all MeloTTS models from VRAM."""
    if lang_code is None:
        for k in list(MELO.keys()):
            free_melo(k)
        return
    if lang_code in MELO and MELO[lang_code] is not None:
        del MELO[lang_code]
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f'[Free] {lang_code!r} MeloTTS released', flush=True)


# ── Tone color (speaker embedding) extraction ───────────────────
def extract_target_se(ref_audio_path: str, version: str = 'v2', vad: bool = True):
    """Extract the target speaker's tone color embedding from a reference audio clip.
    version: 'v1' | 'v2'
    vad: True to use Silero VAD (default; ~1 GB faster-whisper download on first call)
         False to use whisper-timestamped (heavier, splits on words)
    """
    conv = get_converter(version)
    from openvoice import se_extractor
    return se_extractor.get_se(ref_audio_path, conv, vad=vad)


def load_base_se(lang_code: str, speaker_key: str, version: str = 'v2') -> torch.FloatTensor:
    """Load a pre-computed base-speaker SE for one of the bundled voices.
    speaker_key: lowercase stem like 'en-newest', 'en-default', 'es', 'fr', 'zh', 'jp', 'kr'
    """
    if version == 'v1':
        path = os.path.join(V1_CKPT_DIR, 'base_speakers', 'ses', f'{speaker_key}.pth')
    else:
        path = os.path.join(V2_CKPT_DIR, 'base_speakers', 'ses', f'{speaker_key}.pth')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Base SE not found: {path}')
    return torch.load(path, map_location=DEVICE)


# ── Audio I/O helpers ───────────────────────────────────────────
def save_wav(audio: np.ndarray, sr: int, name: str) -> str:
    path = os.path.join('/content/voice_out', name)
    sf.write(path, audio, sr, subtype='PCM_16')
    return path


def _materialize_audio(audio_obj) -> Tuple[np.ndarray, int]:
    """Gradio can return either a filepath (str) or a (sr, np.ndarray) tuple.
    Returns (np.ndarray, sr)."""
    if isinstance(audio_obj, (list, tuple)):
        if len(audio_obj) == 2 and isinstance(audio_obj[1], np.ndarray):
            return audio_obj[1], int(audio_obj[0])
        return np.asarray(audio_obj[0]), int(audio_obj[1])
    if isinstance(audio_obj, str):
        audio, sr = sf.read(audio_obj)
        return audio, sr
    raise TypeError(f'Unsupported audio input type: {type(audio_obj)}')


# ── Core: voice conversion (audio → audio) ──────────────────────
def convert_voice(src_audio,
                  ref_audio,
                  version: str = 'v2',
                  tau: float = 0.3,
                  watermark_message: str = '@MyShell',
                  progress=None) -> Tuple[np.ndarray, int]:
    """Apply the target voice (from ref_audio) to the source audio.
    src_audio: filepath string OR (sr, ndarray) tuple OR ndarray
    ref_audio: same
    version: 'v1' | 'v2'
    tau: higher (0.5-0.7) = stronger timbre transfer, lower (0.1-0.2) = more source-voice preserved
    Returns: (audio_array, sample_rate)
    """
    if progress:
        progress(0.0, 'Loading converter...')

    # Materialize inputs to file paths so OpenVoice's se_extractor + librosa can read them
    src_path = _ensure_wav_path(src_audio, 'src')
    ref_path = _ensure_wav_path(ref_audio, 'ref')

    if progress:
        progress(0.2, 'Extracting target tone color...')
    target_se, _ = extract_target_se(ref_path, version=version, vad=True)

    # Source SE: pull from the bundled base speakers (whichever language matches src_path)
    # We pick the first available SE; the converter can take any reference and produce a
    # timbre-converted output. If src is your own audio (not a MeloTTS output), we fall
    # back to the "en-default" base SE as a neutral starting point.
    src_lang, src_speaker = _guess_src_lang_speaker(src_path, version)
    if progress:
        progress(0.4, f'Loading source SE ({src_speaker})...')
    src_se = load_base_se(src_lang, src_speaker, version=version)

    if progress:
        progress(0.6, 'Converting voice...')
    t0 = time.time()
    conv = get_converter(version)
    audio_out = conv.convert(
        audio_src_path=src_path,
        src_se=src_se,
        tgt_se=target_se,
        output_path=None,
        tau=tau,
        message=watermark_message,
    )
    if progress:
        progress(1.0, f'Done in {time.time()-t0:.1f}s')
    return np.asarray(audio_out, dtype=np.float32), SAMPLE_RATE


# ── Core: TTS then convert (text → audio, with target voice) ─────
def tts_then_convert(text: str,
                     ref_audio,
                     lang_code: str = 'EN',
                     speaker: str = 'EN-US',
                     speed: float = 1.0,
                     version: str = 'v2',
                     tau: float = 0.3,
                     watermark_message: str = '@MyShell',
                     progress=None) -> Tuple[np.ndarray, int, dict]:
    """Synthesize text in lang_code via MeloTTS, then apply the target voice.
    speaker: e.g. 'EN-US', 'EN-BR', 'EN_INDIA', 'EN-AU', 'EN-Default', 'ES', 'FR', 'ZH', 'JP', 'KR'
    Returns: (audio_array, sample_rate, metadata dict)
    """
    if progress:
        progress(0.0, 'Loading MeloTTS...')
    melo = get_melo(lang_code)
    speaker_ids = melo.hps.data.spk2id
    if speaker not in speaker_ids:
        raise ValueError(f'Unknown speaker {speaker!r} for lang {lang_code!r}. Available: {list(speaker_ids.keys())}')
    speaker_id = speaker_ids[speaker]

    # MeloTTS writes to file; we read it back
    src_path = '/content/voice_out/_melo_tmp.wav'
    if progress:
        progress(0.2, f'MeloTTS generating ({lang_code}/{speaker})...')
    t0 = time.time()
    melo.tts_to_file(text, speaker_id, src_path, speed=speed)
    tts_time = time.time() - t0

    # Apply the target voice
    if progress:
        progress(0.6, 'Extracting target tone color...')
    ref_path = _ensure_wav_path(ref_audio, 'ref')
    target_se, _ = extract_target_se(ref_path, version=version, vad=True)

    # Source SE: derived from the speaker we just used
    src_speaker_key = speaker.lower().replace('_', '-')
    if progress:
        progress(0.7, f'Loading source SE ({src_speaker_key})...')
    src_se = load_base_se(lang_code, src_speaker_key, version=version)

    if progress:
        progress(0.85, 'Converting voice...')
    t0 = time.time()
    conv = get_converter(version)
    audio_out = conv.convert(
        audio_src_path=src_path,
        src_se=src_se,
        tgt_se=target_se,
        output_path=None,
        tau=tau,
        message=watermark_message,
    )
    convert_time = time.time() - t0

    # Cleanup temp
    try:
        os.remove(src_path)
    except Exception:
        pass

    if progress:
        progress(1.0, 'Done.')
    return np.asarray(audio_out, dtype=np.float32), SAMPLE_RATE, {
        'tts_time': tts_time,
        'convert_time': convert_time,
        'lang': lang_code,
        'speaker': speaker,
        'version': version,
        'tau': tau,
    }


# ── Helpers for file I/O + source-language guessing ─────────────
def _ensure_wav_path(audio_obj, tag: str) -> str:
    """Write the input audio to a temporary wav file and return its path.
    Gradio can return a filepath string, a (sr, ndarray) tuple, or a raw ndarray.
    """
    if isinstance(audio_obj, str):
        return audio_obj
    audio, sr = _materialize_audio(audio_obj)
    path = f'/content/voice_out/_{tag}_{int(time.time()*1000)}.wav'
    sf.write(path, audio, sr, subtype='PCM_16')
    return path


def _guess_src_lang_speaker(src_audio_path: str, version: str):
    """Heuristic: pick the first bundled base speaker as a neutral source SE.
    OpenVoice's tone converter just needs any valid SE — it doesn't have to
    match the actual content language, because it only uses it as the
    timbre-free starting point.
    """
    return 'EN', 'en-default'


print('[Core] Ready. Use STEP 4 for the UI, STEP 6 for quick test, STEP 7 for batch.')


In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app with six tabs: **Convert** (audio in + ref audio in → audio out), **TTS+Convert** (text + ref audio → audio out), **Style Controls** (emotion/accent/tau), **Batch** (zip of conversions), **VRAM** (free loaded models), **Help** (multilingual notes, citation).

import os, sys, time, json, io, zipfile
import numpy as np
import soundfile as sf
import gradio as gr

# ── Colab / nest_asyncio so Gradio can run inside the notebook loop ──
try:
    import nest_asyncio
    nest_asyncio.apply()
except Exception:
    pass
try:
    from IPython.display import clear_output as _clear
    _clear()
except Exception:
    pass


def _normalize_audio(audio):
    """Gradio can pass filepath or (sr, ndarray). Return (sr, ndarray) or None."""
    if audio is None:
        return None
    if isinstance(audio, (list, tuple)):
        if len(audio) == 2 and isinstance(audio[1], np.ndarray):
            return int(audio[0]), audio[1]
        return None
    if isinstance(audio, str):
        data, sr = sf.read(audio)
        return int(sr), np.asarray(data, dtype=np.float32)
    return None


# ── Tab 1: Voice Convert ────────────────────────────────────────
def ui_convert(src_audio, ref_audio, version, tau, watermark, progress=gr.Progress()):
    if not ref_audio:
        raise gr.Error('Upload a reference audio (5-30s of clear speech).')
    if not src_audio:
        raise gr.Error('Upload a source audio to convert.')
    try:
        audio, sr = convert_voice(src_audio, ref_audio, version, tau, watermark, progress)
    except Exception as e:
        raise gr.Error(f'Conversion failed: {e}')
    fname = f'convert_{int(time.time())}.wav'
    path = save_wav(audio, sr, fname)
    meta = (
        f"version = {version} · tau = {tau:.2f} · watermark = '{watermark}'\n"
        f"duration = {len(audio)/sr:.2f}s · sample_rate = {sr} Hz\n"
        f"src = {os.path.basename(_ensure_wav_path(src_audio, 'x'))} · "
        f"ref = {os.path.basename(_ensure_wav_path(ref_audio, 'x'))}\n"
    )
    return (sr, audio), path, meta


# ── Tab 2: TTS + Convert ────────────────────────────────────────
LANG_LABEL_TO_CODE = {label: code for code, label, _ in MELO_LANGS}


def ui_tts_convert(text, ref_audio, lang_label, speaker, speed, version, tau,
                   watermark, progress=gr.Progress()):
    if not text or not text.strip():
        raise gr.Error('Type some text first.')
    if not ref_audio:
        raise gr.Error('Upload a reference audio (5-30s of clear speech).')
    code = LANG_LABEL_TO_CODE.get(lang_label, 'EN')
    try:
        audio, sr, meta = tts_then_convert(
            text.strip(), ref_audio, code, speaker,
            float(speed), version, float(tau), watermark, progress,
        )
    except Exception as e:
        raise gr.Error(f'Generation failed: {e}')
    fname = f'tts_convert_{int(time.time())}.wav'
    path = save_wav(audio, sr, fname)
    info = (
        f"version = {version} · lang = {code} · speaker = {speaker} · "
        f"speed = {speed:.2f} · tau = {tau:.2f}\n"
        f"duration = {len(audio)/sr:.2f}s · sample_rate = {sr} Hz\n"
        f"tts = {meta['tts_time']:.1f}s · convert = {meta['convert_time']:.1f}s\n"
    )
    return (sr, audio), path, info


# ── Tab 3: Style Controls ───────────────────────────────────────
def ui_style_explain():
    return (
        "**Style controls in OpenVoice**\n\n"
        "OpenVoice exposes three style levers:\n"
        "- **tau** (0.1 - 0.7, default 0.3) — controls how strongly the target tone color overrides the source. "
        "Lower = source prosody is preserved more; higher = target voice dominates.\n"
        "- **emotion / accent / rhythm / pitch** — these come from the **source audio's** delivery, not from sliders. "
        "OpenVoice transfers *timbre*; it does not synthesize emotion from text. If you want a sad delivery, "
        "use a source audio that's sad. If you want a British accent, use a source that's British.\n"
        "- **watermark message** — embedded 32-bit string at 16 kbps via `wavmark`. Default `@MyShell`. "
        "Disable by passing an empty string.\n\n"
        "**Cross-lingual style transfer**\n\n"
        "The killer feature: feed an English source clip to a Spanish reference voice and get Spanish-accented "
        "English (or vice versa, via the TTS tab). V2 + MeloTTS is the only open-source setup that does this "
        "with native multilingual support for ES/FR/ZH/JA/KR."
    )


# ── Tab 4: Batch ───────────────────────────────────────────────
def ui_batch(src_dir, ref_audio, version, tau, watermark, progress=gr.Progress()):
    """Convert every .wav/.mp3/.flac/.ogg/.m4a in src_dir using the same ref audio."""
    if not ref_audio:
        raise gr.Error('Upload a reference audio.')
    if not src_dir or not os.path.isdir(src_dir):
        raise gr.Error('Provide a directory of source audio files.')
    exts = ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.opus')
    files = sorted(f for f in os.listdir(src_dir) if f.lower().endswith(exts))
    if not files:
        raise gr.Error(f'No audio files found in {src_dir} (looked for {exts})')
    out_dir = '/content/voice_out/batch'
    os.makedirs(out_dir, exist_ok=True)
    log = []
    zip_buf = io.BytesIO()
    with zipfile.ZipFile(zip_buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, fname in enumerate(files):
            src_path = os.path.join(src_dir, fname)
            try:
                audio, sr = convert_voice(src_path, ref_audio, version, tau, watermark)
                out_name = f'{i:03d}__{os.path.splitext(fname)[0]}.wav'
                out_path = os.path.join(out_dir, out_name)
                sf.write(out_path, audio, sr, subtype='PCM_16')
                with open(out_path, 'rb') as f:
                    zf.writestr(out_name, f.read())
                log.append(f'[{i+1:03d}/{len(files)}] ok · {len(audio)/sr:.2f}s · {fname}')
            except Exception as e:
                log.append(f'[{i+1:03d}/{len(files)}] FAILED · {e} · {fname}')
            progress((i+1) / max(len(files), 1), f'{i+1}/{len(files)}')
    zip_path = '/content/voice_out/batch.zip'
    with open(zip_path, 'wb') as f:
        f.write(zip_buf.getvalue())
    return zip_path, '\n'.join(log)


# ── Tab 5: VRAM ─────────────────────────────────────────────────
def ui_vram_list():
    convs = ', '.join(sorted(CONVERTERS.keys())) or '(none)'
    melos = ', '.join(sorted(MELO.keys())) or '(none)'
    if not torch.cuda.is_available():
        return f'Converters: {convs}\nMeloTTS: {melos}\n(CPU mode — no GPU memory to query.)'
    free, tot = torch.cuda.mem_get_info(0)
    return (f'Converters: {convs}\nMeloTTS: {melos}\n'
            f'GPU memory: {free/1e9:.1f} / {tot/1e9:.1f} GB free')


def ui_free_all():
    free_converter(None)
    free_melo(None)
    if torch.cuda.is_available():
        free, tot = torch.cuda.mem_get_info(0)
        return f'Freed all.\nGPU memory: {free/1e9:.1f} / {tot/1e9:.1f} GB free.'
    return 'Freed all. (CPU mode — no GPU to query.)'


# ── Build the UI ────────────────────────────────────────────────
with gr.Blocks(title='OpenVoice V2', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# OpenVoice V2 — Instant Voice Cloning & Conversion\n'
                'MIT licensed · 36.6k★ · myshell-ai/OpenVoice')

    with gr.Tabs():
        # ── Tab 1: Convert ──
        with gr.Tab('Convert'):
            gr.Markdown('**Change the voice of an audio clip** while keeping the words and prosody. '
                        'Upload a source audio and a short reference clip (5-30 s) of the target voice.')
            with gr.Row():
                with gr.Column():
                    c_src = gr.Audio(label='Source audio (the words you want to say)',
                                     type='numpy',
                                     info='Any language. The content/prosody is preserved; only the timbre changes.')
                    c_ref = gr.Audio(label='Reference voice (5-30s of clear speech)',
                                     type='numpy',
                                     info='The voice you want the output to sound like. No transcript needed.')
                    c_version = gr.Dropdown(['v2', 'v1'], value='v2', label='Model version',
                                            info="V2 = multilingual + better quality. V1 = simpler, EN/ZH only.")
                    c_tau = gr.Slider(0.1, 0.7, value=0.3, step=0.05, label='Tone transfer strength (tau)',
                                      info='Higher = target voice dominates. Lower = source prosody is preserved.')
                    c_wm = gr.Textbox(label='Watermark message', value='@MyShell',
                                      info='32-bit string embedded in audio via wavmark. Leave empty to disable.')
                    c_btn = gr.Button('🎭 Convert voice', variant='primary')
                with gr.Column():
                    c_audio = gr.Audio(label='Output', type='numpy',
                                       info='Click play to listen. Use the download icon to save the .wav.')
                    c_file = gr.File(label='Download .wav')
                    c_meta = gr.Textbox(label='Run metadata', lines=5, interactive=False)

            c_btn.click(ui_convert, [c_src, c_ref, c_version, c_tau, c_wm],
                        [c_audio, c_file, c_meta])

        # ── Tab 2: TTS + Convert ──
        with gr.Tab('TTS + Convert'):
            gr.Markdown('**Type text in any supported language, get it spoken in the reference voice.** '
                        'MeloTTS synthesizes the content; OpenVoice applies the target tone color.')
            with gr.Row():
                with gr.Column():
                    t_text = gr.Textbox(label='Text', lines=4,
                                         value='Did you ever hear a folk tale about a giant turtle?',
                                         info='MeloTTS will synthesize this in the chosen language.')
                    t_lang = gr.Dropdown([label for _, label, _ in MELO_LANGS],
                                          value='English 🇺🇸 (newest)', label='Language',
                                          info='Picks the MeloTTS base speaker.')
                    t_speaker = gr.Dropdown(
                        # Populated lazily when lang changes
                        choices=["EN-US", "EN-Default", "EN-BR", "EN_INDIA", "EN-AU"],
                        value='EN-US', label='Speaker',
                        info='Built-in MeloTTS voice. List updates when you change Language.')
                    t_speed = gr.Slider(0.5, 2.0, value=1.0, step=0.05, label='Speed',
                                         info='Speech rate multiplier. 1.0 = normal.')
                    t_ref = gr.Audio(label='Reference voice (5-30s of clear speech)',
                                     type='numpy',
                                     info='The voice you want the TTS output to sound like.')
                    t_version = gr.Dropdown(['v2', 'v1'], value='v2', label='Model version',
                                             info='V2 = multilingual + better quality.')
                    t_tau = gr.Slider(0.1, 0.7, value=0.3, step=0.05, label='Tone transfer strength (tau)',
                                       info='Higher = target voice dominates. Lower = source prosody preserved.')
                    t_wm = gr.Textbox(label='Watermark message', value='@MyShell',
                                       info='32-bit string embedded in audio. Empty = disabled.')
                    t_btn = gr.Button('🗣️ Generate', variant='primary')
                with gr.Column():
                    t_audio = gr.Audio(label='Output', type='numpy',
                                        info='Click play to listen. Use the download icon to save.')
                    t_file = gr.File(label='Download .wav')
                    t_info = gr.Textbox(label='Run info', lines=5, interactive=False)

            def _speakers_for_lang(lang_label):
                code = LANG_LABEL_TO_CODE.get(lang_label, 'EN')
                # Standard sets per language; lazy
                if code.startswith('EN'):
                    return ["EN-US", "EN-Default", "EN-BR", "EN_INDIA", "EN-AU"]
                speakers = {
                    'ES': ["ES"], 'FR': ["FR"], 'ZH': ["ZH"],
                    'JP': ["JP"], 'KR': ["KR"],
                }
                return speakers.get(code, [code])

            t_lang.change(_speakers_for_lang, inputs=t_lang, outputs=t_speaker)
            t_btn.click(ui_tts_convert,
                        [t_text, t_ref, t_lang, t_speaker, t_speed, t_version, t_tau, t_wm],
                        [t_audio, t_file, t_info])

        # ── Tab 3: Style Controls ──
        with gr.Tab('Style Controls'):
            gr.Markdown(ui_style_explain())

        # ── Tab 4: Batch ──
        with gr.Tab('Batch'):
            gr.Markdown('**Convert every audio file in a directory** using the same reference voice. '
                        'Useful for dubbing, character voicing, or batch translation workflows.')
            with gr.Row():
                with gr.Column():
                    b_src = gr.Textbox(label='Source directory (path on this Colab VM)',
                                        value='/content/voice_out/batch_in',
                                        info='Mount Drive first if your files live there. Looks for .wav/.mp3/.flac/.ogg/.m4a/.opus.')
                    b_ref = gr.Audio(label='Reference voice', type='numpy',
                                     info='One ref voice used for every file in the directory.')
                    b_version = gr.Dropdown(['v2', 'v1'], value='v2', label='Model version')
                    b_tau = gr.Slider(0.1, 0.7, value=0.3, step=0.05, label='Tone transfer strength (tau)')
                    b_wm = gr.Textbox(label='Watermark message', value='@MyShell')
                    b_btn = gr.Button('📦 Run batch', variant='primary')
                with gr.Column():
                    b_zip = gr.File(label='Download .zip of converted files')
                    b_log = gr.Textbox(label='Per-file log', lines=10, interactive=False)

            b_btn.click(ui_batch, [b_src, b_ref, b_version, b_tau, b_wm], [b_zip, b_log])

        # ── Tab 5: VRAM ──
        with gr.Tab('VRAM'):
            gr.Markdown('OpenVoice V2 is small (~140 MB) and MeloTTS is ~150 MB per language. '
                        'Total VRAM budget is ~1-2 GB if you load all 7 languages.')
            v_log = gr.Textbox(label='Loaded models', lines=4, interactive=False)
            v_btn = gr.Button('🧹 Free all loaded models', variant='secondary')
            v_btn.click(ui_free_all, outputs=v_log)
            demo.load(ui_vram_list, outputs=v_log)

        # ── Tab 6: Help ──
        with gr.Tab('Help'):
            gr.Markdown(
                """## Multilingual support

OpenVoice V2 supports 7 languages via MeloTTS as the base speaker:

| Code | Language | Built-in speakers |
| --- | --- | --- |
| EN_NEWEST | English 🇺🇸 (newest) | EN-US, EN-Default |
| EN | English 🇺🇸 (default) | EN-US, EN-Default, EN-BR, EN_INDIA, EN-AU |
| ES | Spanish 🇪🇸 | ES |
| FR | French 🇫🇷 | FR |
| ZH | Chinese 🇨🇳 (mixed EN) | ZH |
| JP | Japanese 🇯🇵 | JP |
| KR | Korean 🇰🇷 | KR |

Note: **V1 supports only English and Chinese**. Switch to V2 in the UI for full multilingual.

## Watermarking

Every output is watermarked with a 32-bit string at 16 kbps via [`wavmark`](https://github.com/wavmark/wavmark). The default is `@MyShell`. To disable, set the Watermark field to an empty string. The watermark survives MP3 compression at 128 kbps and most common audio editing.

To detect a watermark, use the `Watermark decoder` tab in the [official OpenVoice Space](https://huggingface.co/spaces/myshell-ai/OpenVoiceV2).

## Reference audio tips

- **Length**: 5-30 s is the sweet spot. Shorter than 5 s has unreliable timbre; longer than 30 s doesn't help.
- **Quality**: clean speech, no background music, no reverb. Phone call recordings are bad; close-mic studio recordings are good.
- **Emotion**: the reference's emotion is NOT transferred (only timbre is). The output's emotion comes from the source audio (Convert tab) or is neutral (TTS+Convert tab).
- **VAD trimming**: by default, OpenVoice runs Silero VAD on the reference to keep only speech segments. This is the right call 99% of the time.

## Differences from RVC / SoVITS

| | OpenVoice V2 | RVC / SoVITS |
| --- | --- | --- |
| License | MIT | AGPL-3.0 |
| Multilingual | EN/ES/FR/ZH/JA/KR | EN/ZH/JP mostly |
| Zero-shot | Yes | Yes (with caveats) |
| Real-time | No | Yes (RVC) |
| Training required | No | Optional fine-tune |
| Singing VC | No | Yes (SoVITS) |

## License

[MIT](https://github.com/myshell-ai/OpenVoice/blob/main/LICENSE) — free for commercial use.

## Citation

```bibtex
@article{qin2023openvoice,
  title  = {OpenVoice: Versatile Instant Voice Cloning},
  author = {Qin, Zengyi and Zhao, Wenliang and Yu, Xumin and Sun, Xin},
  journal= {arXiv preprint arXiv:2312.01479},
  year   = {2023}
}
```
"""
            )

# ── Welcome message on first load ──
def _welcome():
    return ('🎭 OpenVoice V2 ready. 7 languages via MeloTTS, MIT licensed. '
            'Start with the **Convert** tab (any audio → any voice) or **TTS+Convert** (text → cloned voice).')

demo.load(_welcome, outputs=[])

demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name='0.0.0.0', server_port=7860,
)


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')


In [ ]:
# @title Step 6 — Quick Test (one-off conversion, no UI)
# @markdown Run a single voice-conversion call from the notebook. Useful for scripting or quick checks.

VERSION = "v2"  # @param ["v1", "v2"]
TAU = 0.3  # @param {type:"slider", min:0.1, max:0.7, step:0.05}
SRC_AUDIO = ""  # @param {type:"string"}
REF_AUDIO = ""  # @param {type:"string"}
WATERMARK = "@MyShell"  # @param {type:"string"}

import os, time
if not SRC_AUDIO or not os.path.exists(SRC_AUDIO):
    raise SystemExit('SRC_AUDIO is required (path to a .wav/.mp3/.flac file).')
if not REF_AUDIO or not os.path.exists(REF_AUDIO):
    raise SystemExit('REF_AUDIO is required (path to a .wav/.mp3/.flac file).')

print(f'[Test] version={VERSION} tau={TAU:.2f} src={os.path.basename(SRC_AUDIO)} ref={os.path.basename(REF_AUDIO)}')
t0 = time.time()
audio, sr = convert_voice(SRC_AUDIO, REF_AUDIO, VERSION, TAU, WATERMARK)
elapsed = time.time() - t0
path = save_wav(audio, sr, f'quicktest_{int(time.time())}.wav')
print(f'\n[Done] {path}')
print(f'[Done] {len(audio)} samples · {len(audio)/sr:.2f}s @ {sr} Hz · {elapsed:.1f}s wall')

from IPython.display import Audio, display
display(Audio(path, autoplay=False))


In [ ]:
# @title Step 7 — Batch Conversion
# @markdown Convert every audio file in a directory using the same reference voice. Each input becomes one output .wav, plus a zip of all results.

VERSION = "v2"  # @param ["v1", "v2"]
TAU = 0.3  # @param {type:"slider", min:0.1, max:0.7, step:0.05}
REF_AUDIO = ""  # @param {type:"string"}
SRC_DIR = "/content/voice_out/batch_in"  # @param {type:"string"}
OUT_DIR = "/content/voice_out/batch"  # @param {type:"string"}
WATERMARK = "@MyShell"  # @param {type:"string"}

import os, time
if not REF_AUDIO or not os.path.exists(REF_AUDIO):
    raise SystemExit('REF_AUDIO is required (path to a .wav/.mp3/.flac file).')
if not SRC_DIR or not os.path.isdir(SRC_DIR):
    raise SystemExit(f'SRC_DIR is required and must be a directory (got {SRC_DIR!r}).')

exts = ('.wav', '.mp3', '.flac', '.ogg', '.m4a', '.opus')
files = sorted(f for f in os.listdir(SRC_DIR) if f.lower().endswith(exts))
os.makedirs(OUT_DIR, exist_ok=True)
print(f'[Batch] {len(files)} files in {SRC_DIR} · version={VERSION} tau={TAU:.2f}')

t_start = time.time()
for i, fname in enumerate(files):
    src_path = os.path.join(SRC_DIR, fname)
    label = f"[{i+1:03d}/{len(files)}]"
    print(f'{label} {fname}', flush=True)
    t0 = time.time()
    try:
        audio, sr = convert_voice(src_path, REF_AUDIO, VERSION, TAU, WATERMARK)
        out_name = f'{i:03d}__{os.path.splitext(fname)[0]}.wav'
        out_path = os.path.join(OUT_DIR, out_name)
        sf.write(out_path, audio, sr, subtype='PCM_16')
        print(f'  ✓ {len(audio)/sr:.2f}s · {time.time()-t0:.1f}s · {out_path}', flush=True)
    except Exception as e:
        print(f'  ✗ EXCEPTION: {e}', flush=True)
        continue

elapsed = time.time() - t_start
print(f'\n[Done] {len(files)} files in {OUT_DIR} · total wall time {elapsed:.1f}s')
